In [2]:
import gymnasium as gym
import numpy as np
import time

In [8]:
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode='human')
state, info = env.reset()

done = False
while not done:
    action = env.action_space.sample() # takes a random raction
    print(f"Current State: {state}, Taking Action: {action}")

    new_state , reward , terminated , truncated  , info = env.step(action)
    done = terminated or truncated # An episode ends if terminated (win/loss) or truncated (timeout)

    env.render()

    state = new_state

    if reward > 0:
        print("Success! Reached the Goal!")
    time.sleep(0.5)

env.close()
print("Episode finished.")
    
# 0: Left, 1: Down, 2: Right, 3: Up
# For now i am picking random actions

Current State: 0, Taking Action: 0
Current State: 0, Taking Action: 0
Current State: 0, Taking Action: 3
Current State: 0, Taking Action: 1
Current State: 4, Taking Action: 2
Episode finished.


Agent Environment loop ->
It is the loop like first(reset):intial state  -> loop(action -> step):in loop agent takes action and takes <br>
a step to get new reward and new state.<br>
Basically rl is like that :: intital state -> loop(action -> next state -> reward) <br>
So Lets assume Tesla : First we we define a agent that will be made by ourself, maybe by some ml algorithm then <br>
we will do some action like [buy , sell , hold ] and after that we will get the next state and it will go in loop and we satified our<br>
say we achieved our rewards[up , down] then we will exit out of the loop.

State-> again assuming case of TSLA : [current_price, volume, moving_average_10, RSI_14, etc.]
Actions -> we can buy , sell , hold these will count towards action.
Reward fucntion -> reward function can be risky as lets say when we are doing some selling of stocks and we get the got numbers means a reward <br> which we set in our fucntion then it will seel but what we didn't see that it might can go more which we didn't even wait.


In [9]:
# run_cartpole.py
import gymnasium as gym
from stable_baselines3 import A2C

env = gym.make("CartPole-v1", render_mode="human")

model = A2C("MlpPolicy", env, verbose=1)
print("--- Training model ---")
model.learn(total_timesteps=10000)
print("--- Training finished ---")

vec_env = model.get_env()
obs = vec_env.reset()
print("\n--- Testing trained model ---")
for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    # The render call is now inside the loop to see the agent act
    # vec_env.render() is the new way for SB3
    if done.any():
        print("Episode finished. Resetting.")
        obs = vec_env.reset()

env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
--- Training model ---
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 45       |
|    ep_rew_mean        | 45       |
| time/                 |          |
|    fps                | 45       |
|    iterations         | 100      |
|    time_elapsed       | 10       |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -0.634   |
|    explained_variance | -0.542   |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -4.32    |
|    value_loss         | 74.4     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 44       |
|    ep_rew_mean        | 44       |
| time/                 |          |
|    fps                | 46       |
|    iterations         | 200   



The script you ran performs the full, end-to-end process of a basic RL project in just a few lines of code, thanks to the `stable-baselines3` library.

1.  **`env = gym.make(...)`**: You created an instance of the CartPole world. This is your **Environment**. Its job is to keep track of the cart's position, the pole's angle, and to tell the agent when the pole has fallen over.
2.  **`model = A2C("MlpPolicy", env, verbose=1)`**: This is the most important line. You created your **Agent**.
    *   `A2C`: You told the agent to use the "Advantage Actor-Critic" algorithm. This is a powerful, standard RL algorithm.
    *   `"MlpPolicy"`: You specified that the agent's "brain" should be a **M**ulti-**L**ayer **P**erceptron—a standard type of neural network.
    *   `verbose=1`: You told the model to be "talkative" and print its training progress, which is the table you see.
3.  **`model.learn(total_timesteps=10000)`**: This is the command that starts the training. You told the agent: "Interact with the CartPole environment for 10,000 time steps. For every action you take, observe the outcome and update your neural network to get better." This single line encapsulates the entire agent-environment loop and the neural network's learning process.

### Part 2: What The Output Table Means

This table is the "cockpit display" for your agent's training flight. It tells you everything about its performance and the learning process. Let's go section by section.

#### `rollout/`
This section describes the agent's actual performance in the environment. This is what tells you if it's learning to do its job.

*   **`ep_len_mean` (Episode Length Mean):** 45
    *   **Meaning:** On average, your agent's episodes are lasting for 45 steps before the pole falls. In CartPole, the goal is to last as long as possible.
*   **`ep_rew_mean` (Episode Reward Mean):** 45
    *   **Meaning:** This is the **single most important metric here.** In CartPole, the agent gets a +1 reward for every single time step it keeps the pole balanced. So, this means your agent is averaging a score of 45 per attempt. As training continues, you would expect this number to go up significantly (towards 500, the maximum).

#### `time/`
This section describes the computational performance of the training.

*   **`fps` (Frames Per Second):** 45
    *   **Meaning:** Your computer is running the simulation and training at a rate of 45 steps per second.
*   **`iterations`:** 100
    *   **Meaning:** The model has gone through 100 learning updates.
*   **`time_elapsed`:** 10
    *   **Meaning:** 10 seconds have passed since training started.
*   **`total_timesteps`:** 500
    *   **Meaning:** The agent has taken a total of 500 actions in the environment so far.

#### `train/`
This section shows the internal state of the neural network's learning process. These are diagnostic metrics.

*   **`entropy_loss`:** -0.634
    *   **Meaning:** This relates to how much the agent is "exploring" (trying random actions). It typically starts high and decreases as the agent becomes more confident in its strategy.
*   **`explained_variance`:** -0.542
    *   **Meaning:** A measure of how well the agent's internal "value" predictions match the actual rewards it's getting. A value near 1.0 is perfect. A negative value is very poor, which is **completely normal at the very beginning of training.** The agent's brain is still random and hasn't learned anything meaningful yet.
*   **`learning_rate`:** 0.0007
    *   **Meaning:** The current learning rate being used by the optimizer to update the network's weights.
*   **`policy_loss`:** -4.32
    *   **Meaning:** The error associated with the agent's decision-making part (the "Actor"). The agent tries to minimize this to make better decisions.
*   **`value_loss`:** 74.4
    *   **Meaning:** The error associated with the agent's state-evaluation part (the "Critic"). The agent tries to minimize this to get better at judging how "good" a situation is.

---
